# Model B — Lightweight CNN (Depthwise Separable Convolutions)
**EN3150 Pattern Recognition — Assignment 03**

This notebook implements a custom lightweight CNN for Fashion-MNIST. Model B replaces standard spatial convolutions with **depthwise separable convolutions** and checks that the trainable parameter count is below the assignment limit of **100,000**.

**Important:** To make the comparison fair, this notebook tries to load the same processed dataset used by Model A. Review the dataset-loading cell and update the paths/keys if your preprocessing notebook saved the data under a different name. Do not use the test set for model selection.

In [1]:
import os
import time
import json
import random
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

TensorFlow: 2.21.0
Num GPUs Available: 0


## 2. Configuration

Set the dataset path and training options here. The assignment requires images no larger than **64 × 64**, a **70/15/15** train/validation/test split, and at least **20 epochs** of training.


In [2]:
DATA_PATH = Path("../data/processed/fashion_mnist_processed.npz")
OUTPUT_DIR = Path("outputs/model_b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 64
BATCH_SIZE = 128
EPOCHS = 25                  # at least 20 epochs, as required
NUM_CLASSES = 10
MAX_TRAINABLE_PARAMS = 100_000

# Optimizer comparison: chosen optimizer, plain SGD, and SGD with momentum.
LEARNING_RATE = 1e-3
SGD_LEARNING_RATE = 1e-2
MOMENTUM = 0.9

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

## 3. Load the shared processed dataset

Expected format: an NPZ archive containing separate train, validation, and test arrays. The loader recognizes common key names. If this cell cannot find your arrays, inspect `np.load(DATA_PATH).files` and edit the key mapping below to match the file produced by `01_data_exploration.ipynb`.

This notebook deliberately does **not** silently create a new split if a processed dataset is missing, because that could make Model B use different data from Model A.

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {DATA_PATH.resolve()}. "
        "Run your preprocessing notebook first, then update DATA_PATH to the saved NPZ file."
    )

archive = np.load(DATA_PATH, allow_pickle=False)
print("Available NPZ keys:", archive.files)

KEY_CANDIDATES = {
    "x_train": ["x_train", "X_train", "train_images", "X_train_processed", "x_train_processed"],
    "y_train": ["y_train", "Y_train", "train_labels", "y_train_processed"],
    "x_val":   ["x_val", "X_val", "val_images", "validation_images", "X_val_processed", "x_val_processed"],
    "y_val":   ["y_val", "Y_val", "val_labels", "validation_labels", "y_val_processed"],
    "x_test":  ["x_test", "X_test", "test_images", "X_test_processed", "x_test_processed"],
    "y_test":  ["y_test", "Y_test", "test_labels", "y_test_processed"],
}

def find_key(candidates, available):
    for key in candidates:
        if key in available:
            return key
    return None

resolved = {name: find_key(candidates, archive.files)
            for name, candidates in KEY_CANDIDATES.items()}
missing = [name for name, key in resolved.items() if key is None]
if missing:
    raise KeyError(
        f"Could not find arrays for {missing}. Available keys: {archive.files}. "
        "Update KEY_CANDIDATES to match your saved dataset."
    )

x_train = archive[resolved["x_train"]]
y_train = archive[resolved["y_train"]]
x_val   = archive[resolved["x_val"]]
y_val   = archive[resolved["y_val"]]
x_test  = archive[resolved["x_test"]]
y_test  = archive[resolved["y_test"]]

print("Resolved keys:", resolved)
print("Raw shapes:", x_train.shape, y_train.shape, x_val.shape, y_val.shape, x_test.shape, y_test.shape)

Available NPZ keys: ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']
Resolved keys: {'x_train': 'X_train', 'y_train': 'y_train', 'x_val': 'X_val', 'y_val': 'y_val', 'x_test': 'X_test', 'y_test': 'y_test'}
Raw shapes: (49000, 28, 28, 1) (49000,) (10500, 28, 28, 1) (10500,) (10500, 28, 28, 1) (10500,)


### 3.1 Normalize labels and image arrays

Images are converted to float32 and normalized to `[0, 1]` if their values look like 8-bit pixel values. Grayscale images are expanded to one channel. Images larger than 64 × 64 are resized down to 64 × 64. Labels may be integer class IDs or one-hot encoded vectors.

In [5]:
def prepare_images(images, image_size=IMAGE_SIZE):
    images = np.asarray(images)
    if images.ndim == 3:  # (N, H, W)
        images = images[..., np.newaxis]
    if images.ndim != 4:
        raise ValueError(f"Expected images with shape (N,H,W) or (N,H,W,C); got {images.shape}")

    images = images.astype(np.float32)
    if np.nanmax(images) > 1.0:
        images = images / 255.0

    if images.shape[1] > image_size or images.shape[2] > image_size:
        images = tf.image.resize(images, (image_size, image_size), method="bilinear").numpy()
    return images

def prepare_labels(labels):
    labels = np.asarray(labels)
    if labels.ndim > 1 and labels.shape[-1] == NUM_CLASSES:
        labels = np.argmax(labels, axis=-1)
    return labels.astype(np.int64).reshape(-1)

x_train = prepare_images(x_train)
x_val   = prepare_images(x_val)
x_test  = prepare_images(x_test)
y_train = prepare_labels(y_train)
y_val   = prepare_labels(y_val)
y_test  = prepare_labels(y_test)

assert x_train.shape[0] == y_train.shape[0]
assert x_val.shape[0] == y_val.shape[0]
assert x_test.shape[0] == y_test.shape[0]
assert max(x_train.shape[1:3]) <= 64
assert max(x_val.shape[1:3]) <= 64
assert max(x_test.shape[1:3]) <= 64
assert set(np.unique(np.concatenate([y_train, y_val, y_test]))).issubset(set(range(NUM_CLASSES)))

print("Prepared shapes:")
print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)
print("Pixel range:", float(x_train.min()), "to", float(x_train.max()))
print("Split percentages:", *(round(100 * n / (len(x_train)+len(x_val)+len(x_test)), 2)
                              for n in [len(x_train), len(x_val), len(x_test)]))

Prepared shapes:
Train: (49000, 28, 28, 1) (49000,)
Validation: (10500, 28, 28, 1) (10500,)
Test: (10500, 28, 28, 1) (10500,)
Pixel range: 0.0 to 1.0
Split percentages: 70.0 15.0 15.0


## 4. Define Model B

Each block uses a `DepthwiseConv2D` followed by a `1 × 1` `Conv2D`. ReLU is used for an inexpensive activation. Global average pooling avoids a large flatten layer and helps keep the parameter count small. The final layer uses softmax for the 10 Fashion-MNIST classes.

In [6]:
def depthwise_separable_block(x, filters, stride=1, block_name="block"):
    x = tf.keras.layers.DepthwiseConv2D(
        kernel_size=3,
        strides=stride,
        padding="same",
        use_bias=False,
        name=f"{block_name}_depthwise",
    )(x)
    x = tf.keras.layers.BatchNormalization(name=f"{block_name}_dw_bn")(x)
    x = tf.keras.layers.ReLU(name=f"{block_name}_dw_relu")(x)

    x = tf.keras.layers.Conv2D(
        filters=filters,
        kernel_size=1,
        strides=1,
        padding="same",
        use_bias=False,
        name=f"{block_name}_pointwise",
    )(x)
    x = tf.keras.layers.BatchNormalization(name=f"{block_name}_pw_bn")(x)
    x = tf.keras.layers.ReLU(name=f"{block_name}_pw_relu")(x)
    return x

def build_model_b(input_shape, num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=input_shape, name="image")
    x = tf.keras.layers.Conv2D(
        16, kernel_size=3, padding="same", use_bias=False, name="stem_conv"
    )(inputs)
    x = tf.keras.layers.BatchNormalization(name="stem_bn")(x)
    x = tf.keras.layers.ReLU(name="stem_relu")(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2, name="stem_pool")(x)

    x = depthwise_separable_block(x, 24, stride=1, block_name="ds_block1")
    x = tf.keras.layers.MaxPooling2D(pool_size=2, name="pool1")(x)

    x = depthwise_separable_block(x, 32, stride=1, block_name="ds_block2")
    x = tf.keras.layers.MaxPooling2D(pool_size=2, name="pool2")(x)

    x = depthwise_separable_block(x, 48, stride=1, block_name="ds_block3")
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = tf.keras.layers.Dropout(0.20, name="dropout")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_output")(x)
    return tf.keras.Model(inputs, outputs, name="Model_B_Lightweight_CNN")

input_shape = tuple(x_train.shape[1:])
model_b = build_model_b(input_shape)
model_b.summary()

trainable_params = int(np.sum([np.prod(v.shape) for v in model_b.trainable_variables]))
total_params = model_b.count_params()
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
if trainable_params > MAX_TRAINABLE_PARAMS:
    raise ValueError(
        f"Model B has {trainable_params:,} trainable parameters, exceeding the limit "
        f"of {MAX_TRAINABLE_PARAMS:,}. Reduce the filter counts before training."
    )

Model: "Model_B_Lightweight_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ stem_conv (Conv2D)              │ (None, 28, 28, 16)     │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ stem_bn (BatchNormalization)    │ (None, 28, 28, 16)     │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ stem_relu (ReLU)                │ (None, 28, 28, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ stem_pool (MaxPooling2D)        │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block1_depthwise             │ (None, 14, 14, 16)     │           144 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block1_dw_bn                 │ (None, 14, 14, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block1_dw_relu (ReLU)        │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block1_pointwise (Conv2D)    │ (None, 14, 14, 24)     │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block1_pw_bn                 │ (None, 14, 14, 24)     │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block1_pw_relu (ReLU)        │ (None, 14, 14, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 7, 7, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block2_depthwise             │ (None, 7, 7, 24)       │           216 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block2_dw_bn                 │ (None, 7, 7, 24)       │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block2_dw_relu (ReLU)        │ (None, 7, 7, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block2_pointwise (Conv2D)    │ (None, 7, 7, 32)       │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block2_pw_bn                 │ (None, 7, 7, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block2_pw_relu (ReLU)        │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 3, 3, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block3_depthwise             │ (None, 3, 3, 32)       │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ds_block3_dw_bn                 │ (None, 3, 3, 32)       │           128 │
│ (BatchNormalization)            │                        │             

 Total params: 4,738 (18.51 KB)

 Trainable params: 4,354 (17.01 KB)

 Non-trainable params: 384 (1.50 KB)

Total parameters: 4,738
Trainable parameters: 4,354
